# HPL → NetCDF Production Converter

Converts each `Stare_116_YYYYMMDD_HH.hpl` file under `data/` to a NetCDF-4 file in
`data/netcdf_stare/`.

Per-file output includes all raw beam fields (`dopplervel`, `intensity`, `beta`,
`azimuth`, `elevangle`, `pitch`, `roll`, `height`) plus the SNR-filtered beam-mean
Doppler velocity (`mdv_snr_mean`) and its valid-gate count (`n_mdv_realizations`).

No syncing or motion correction is done here — this is a pure format conversion with
`mdv_snr_mean` computed as a convenience diagnostic variable.

**Cells:**
1. Preamble — imports and load `read_lidar` functions
2. `convert_hpl_to_netcdf` — wrapper function (test / all modes)
3. Test run — 2 files, verify output
4. Production trigger — all 967 files (commented out by default)

In [ ]:
# preamble
using Pkg; Pkg.activate(".")

using Dates
using Printf

includet("./read_lidar.jl")
using .read_lidar: read_streamlinexr_stare,
                   write_stare_hourly_netcdf,
                   read_stare_hourly_netcdf

const HPL_DATA_DIR   = joinpath(@__DIR__, "data")
const NETCDF_OUT_DIR = joinpath(@__DIR__, "data", "netcdf_stare")
const SNR_THRESHOLD  = 1.03

mkpath(NETCDF_OUT_DIR)
println("Output dir: ", NETCDF_OUT_DIR)

: 

In [ ]:
# HPL → NetCDF conversion wrapper
#
# mode=:test  → convert only the files listed in test_files
# mode=:all   → discover and convert all Stare_116_* HPL files under HPL_DATA_DIR
#
# Writes a running CSV log (one row per file) so progress survives interruption
# and can be monitored with:  tail -f data/netcdf_stare/convert_log_*.txt
#
# Returns: (; n_done, n_skip, n_err, errors, log_path)

function find_all_hpl_files(datadir::AbstractString)
    hpl = String[]
    for daydir in readdir(datadir; join=true)
        isdir(daydir) || continue
        for fname in readdir(daydir; join=true)
            occursin(r"^Stare_116_\d{8}_\d{2}\.hpl$", basename(fname)) &&
                push!(hpl, fname)
        end
    end
    sort!(hpl)
end

function convert_hpl_to_netcdf(;
        mode::Symbol           = :test,
        test_files             = [
            joinpath(HPL_DATA_DIR, "20240428", "Stare_116_20240428_00.hpl"),
            joinpath(HPL_DATA_DIR, "20240428", "Stare_116_20240428_01.hpl"),
        ],
        out_dir::String        = NETCDF_OUT_DIR,
        snr_threshold::Float64 = SNR_THRESHOLD,
        overwrite::Bool        = false,
        log_path::String       = joinpath(out_dir,
            "convert_log_$(Dates.format(now(), dateformat"yyyymmdd_HHMMSS")).txt"),
    )

    hpl_files = mode == :all ? find_all_hpl_files(HPL_DATA_DIR) : collect(test_files)
    n_total   = length(hpl_files)
    n_done    = 0
    n_skip    = 0
    n_err     = 0
    errors    = NamedTuple[]

    mkpath(out_dir)
    mkpath(dirname(log_path))

    open(log_path, "w") do log_io
        println(log_io, "iter,filename,status,n_beams,n_valid_mdv,elapsed_s,error_msg")
        flush(log_io)

        @printf("Converting %d HPL files → %s\n", n_total, out_dir)
        overwrite && println("Mode: overwrite=true")

        for (iter, hpl_path) in enumerate(hpl_files)
            stem    = splitext(basename(hpl_path))[1]
            nc_path = joinpath(out_dir, stem * ".nc")

            if !overwrite && isfile(nc_path)
                n_skip += 1
                println(log_io, "$iter,$(basename(hpl_path)),skip,,,0,")
                flush(log_io)
                continue
            end

            t0           = time()
            status       = "ok"
            n_beams      = 0
            n_valid_mdv  = 0
            err_msg      = ""

            try
                beams, header = read_streamlinexr_stare(hpl_path)
                header[:source_hpl] = hpl_path

                n_beams = size(beams[:dopplervel], 1)

                write_stare_hourly_netcdf(nc_path, beams, header;
                                          snr_threshold = snr_threshold)

                # count beams that have at least one valid mdv gate
                int_raw = beams[:intensity]
                n_valid_mdv = count(1:n_beams) do ib
                    any(1:size(int_raw, 2)) do ig
                        iv = int_raw[ib, ig]
                        !ismissing(iv) && coalesce(iv, -Inf) > snr_threshold
                    end
                end

                n_done += 1
            catch err
                status  = "error"
                err_msg = replace(sprint(showerror, err), '\n' => " | ")
                n_err  += 1
                push!(errors, (; iter, hpl_path, err_msg))
                @printf("ERROR [%d/%d] %s\n  %s\n", iter, n_total,
                        basename(hpl_path), err_msg)
            end

            elapsed = round(time() - t0; digits=2)
            println(log_io, "$iter,$(basename(hpl_path)),$status,$n_beams,$n_valid_mdv,$elapsed,$err_msg")
            flush(log_io)

            if mod(iter, 50) == 0 || iter == n_total
                @printf("[%4d/%4d]  done=%-4d skip=%-4d err=%-4d\n",
                        iter, n_total, n_done, n_skip, n_err)
            end
        end
    end

    @printf("\nFinished.  Converted=%d  Skipped=%d  Errors=%d\n  Log: %s\n",
            n_done, n_skip, n_err, log_path)
    return (; n_done, n_skip, n_err, errors, log_path)
end

println("convert_hpl_to_netcdf defined.")

In [ ]:
# TEST: convert 2 files and verify output

conv_test = convert_hpl_to_netcdf(; mode=:test, overwrite=true)

@printf("done=%d  skip=%d  err=%d\n", conv_test.n_done, conv_test.n_skip, conv_test.n_err)

# spot-check the first output file
nc_check = joinpath(NETCDF_OUT_DIR, "Stare_116_20240428_00.nc")
if isfile(nc_check)
    b = read_stare_hourly_netcdf(nc_check)
    mdv_vals = [x for x in b[:mdv_snr_mean] if !ismissing(x) && isfinite(x)]
    @printf("nbeams=%d  valid_mdv=%d  mdv_range=[%.3f, %.3f] m/s\n",
            length(b[:time]), length(mdv_vals),
            isempty(mdv_vals) ? NaN : minimum(mdv_vals),
            isempty(mdv_vals) ? NaN : maximum(mdv_vals))
    @printf("n_real range=[%d, %d]\n",
            minimum(b[:n_mdv_realizations]),
            maximum(b[:n_mdv_realizations]))
    println("snr_threshold=", b[:snr_threshold])
end

In [ ]:
# PRODUCTION: convert all ~967 HPL files (uncomment to run; ~10–20 min)
#
# Skips files already present in data/netcdf_stare/ by default (overwrite=false).
# Monitor progress:  tail -f data/netcdf_stare/convert_log_*.txt

# conv_all = convert_hpl_to_netcdf(; mode=:all, overwrite=false)
# @printf("done=%d  skip=%d  err=%d\n", conv_all.n_done, conv_all.n_skip, conv_all.n_err)
# isempty(conv_all.errors) || println("Errors:\n", join(["  $(e.hpl_path): $(e.err_msg)" for e in conv_all.errors], "\n"))